# Project 01 — Python exploratory analysis

## First-Time Buyer Affordability Pressure by Area

This notebook explores the cleaned area-year affordability dataset and creates chart-ready outputs for the final case study, Excel workbook and Power BI dashboard.


## Analytical focus

The SQL stage established the core rankings and quality checks. This notebook adds exploratory analysis by reviewing distribution, regional variation and five-year movement in affordability pressure.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
PROJECT_ROOT = Path("..").resolve()
CLEANED_DATA = PROJECT_ROOT / "data" / "cleaned" / "affordability_area_year_cleaned.csv"
VISUALS_DIR = PROJECT_ROOT / "visuals"
REPORTS_DIR = PROJECT_ROOT / "reports"

VISUALS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEANED_DATA)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Year range: {int(df['year'].min())} to {int(df['year'].max())}")
df.head()


## Latest-year view


In [ ]:
latest_year = int(df["year"].max())
latest = df.loc[df["year"] == latest_year].copy()
latest_valid = latest.dropna(subset=["lower_quartile_affordability_ratio"]).copy()

print(f"Latest year: {latest_year}")
print(f"Latest-year records: {latest.shape[0]:,}")
print(f"Latest-year records with valid lower-quartile ratio: {latest_valid.shape[0]:,}")


In [ ]:
latest_summary = (
    latest_valid["lower_quartile_affordability_ratio"]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
    .round(2)
    .to_frame(name="lower_quartile_affordability_ratio")
)

latest_summary


## Distribution of latest-year affordability pressure


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(latest_valid["lower_quartile_affordability_ratio"], bins=25)
ax.set_title(f"Distribution of lower-quartile affordability ratios, {latest_year}")
ax.set_xlabel("Lower-quartile affordability ratio")
ax.set_ylabel("Local authority count")
ax.axvline(latest_valid["lower_quartile_affordability_ratio"].median(), linestyle="--")
ax.text(
    latest_valid["lower_quartile_affordability_ratio"].median(),
    ax.get_ylim()[1] * 0.9,
    "Median",
    rotation=90,
    va="top",
)
plt.tight_layout()
distribution_path = VISUALS_DIR / "latest_year_affordability_distribution.png"
plt.savefig(distribution_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {distribution_path.relative_to(PROJECT_ROOT)}")


## Regional comparison


In [ ]:
regional_latest = (
    latest_valid
    .groupby("country_region_name", as_index=False)
    .agg(
        local_authority_count=("local_authority_code", "nunique"),
        median_lq_ratio=("lower_quartile_affordability_ratio", "median"),
        average_lq_ratio=("lower_quartile_affordability_ratio", "mean"),
        max_lq_ratio=("lower_quartile_affordability_ratio", "max"),
    )
    .round(2)
    .sort_values("median_lq_ratio", ascending=False)
)

regional_latest


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
regional_plot = regional_latest.sort_values("median_lq_ratio")
ax.barh(regional_plot["country_region_name"], regional_plot["median_lq_ratio"])
ax.set_title(f"Median lower-quartile affordability ratio by region, {latest_year}")
ax.set_xlabel("Median lower-quartile affordability ratio")
ax.set_ylabel("Country or region")
plt.tight_layout()
regional_path = VISUALS_DIR / "regional_median_affordability_ratio.png"
plt.savefig(regional_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {regional_path.relative_to(PROJECT_ROOT)}")


## Top and bottom latest-year areas


In [ ]:
top_15 = (
    latest_valid
    .sort_values("lower_quartile_affordability_ratio", ascending=False)
    .head(15)
    [[
        "country_region_name",
        "local_authority_name",
        "lower_quartile_house_price",
        "lower_quartile_annual_earnings",
        "lower_quartile_affordability_ratio",
    ]]
)

bottom_15 = (
    latest_valid
    .sort_values("lower_quartile_affordability_ratio", ascending=True)
    .head(15)
    [[
        "country_region_name",
        "local_authority_name",
        "lower_quartile_house_price",
        "lower_quartile_annual_earnings",
        "lower_quartile_affordability_ratio",
    ]]
)

top_15


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
top_plot = top_15.sort_values("lower_quartile_affordability_ratio")
ax.barh(top_plot["local_authority_name"], top_plot["lower_quartile_affordability_ratio"])
ax.set_title(f"Highest first-time buyer affordability pressure, {latest_year}")
ax.set_xlabel("Lower-quartile affordability ratio")
ax.set_ylabel("Local authority")
plt.tight_layout()
top_path = VISUALS_DIR / "top_15_affordability_pressure.png"
plt.savefig(top_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {top_path.relative_to(PROJECT_ROOT)}")


## Five-year change


In [ ]:
comparison_year = latest_year - 5

latest_change = df.loc[df["year"] == latest_year, [
    "local_authority_code",
    "local_authority_name",
    "country_region_name",
    "lower_quartile_affordability_ratio",
]].rename(columns={"lower_quartile_affordability_ratio": "latest_lq_ratio"})

comparison_change = df.loc[df["year"] == comparison_year, [
    "local_authority_code",
    "lower_quartile_affordability_ratio",
]].rename(columns={"lower_quartile_affordability_ratio": "comparison_lq_ratio"})

five_year_change = (
    latest_change
    .merge(comparison_change, on="local_authority_code", how="inner")
    .dropna(subset=["latest_lq_ratio", "comparison_lq_ratio"])
)
five_year_change["five_year_change"] = (
    five_year_change["latest_lq_ratio"] - five_year_change["comparison_lq_ratio"]
)

five_year_change = five_year_change.sort_values("five_year_change", ascending=False)
five_year_change.head(15)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
change_plot = five_year_change.head(15).sort_values("five_year_change")
ax.barh(change_plot["local_authority_name"], change_plot["five_year_change"])
ax.set_title(f"Largest increases in affordability pressure, {comparison_year} to {latest_year}")
ax.set_xlabel("Change in lower-quartile affordability ratio")
ax.set_ylabel("Local authority")
plt.tight_layout()
change_path = VISUALS_DIR / "largest_five_year_increases.png"
plt.savefig(change_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {change_path.relative_to(PROJECT_ROOT)}")


## Save chart-ready summary tables


In [ ]:
regional_latest.to_csv(REPORTS_DIR / "python_regional_latest_summary.csv", index=False)
top_15.to_csv(REPORTS_DIR / "python_top_15_most_pressured.csv", index=False)
bottom_15.to_csv(REPORTS_DIR / "python_bottom_15_least_pressured.csv", index=False)
five_year_change.head(20).to_csv(REPORTS_DIR / "python_top_20_five_year_increases.csv", index=False)
latest_summary.to_csv(REPORTS_DIR / "python_latest_year_distribution_summary.csv")

print("Saved chart-ready summary tables to reports/.")


## Interpretation notes

The exploratory charts should be reviewed alongside the SQL findings. The main narrative to test is whether the latest-year distribution, regional medians and five-year changes support the same business conclusion: affordability pressure is highly localised, with London and the South East most stretched, but recent worsening visible in a broader mix of areas.
